In [1]:
import polars as pl

import nwec.utility_reporting.arrearage_counts
import nwec.utility_reporting.arrearages
import nwec.utils.excel
from nwec.constants import RAW_UTILITY_DATA, Utility

YEAR = 2024
QUARTER = 4
NUM_MONTHS = 3
COLS_PER_MONTH = 1
SHEET_SEARCH_STRING = "past due balances"
ARREARAGE_SEARCH_STRING = "number of customers"
spreadsheet = RAW_UTILITY_DATA / str(YEAR) / f"{Utility.PSE.code}_{YEAR}_Q{QUARTER}.xlsx"
source_date_format = "%Y-%m-%d %H:%M:%S"

In [2]:
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, SHEET_SEARCH_STRING)
df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)
arrearage_counts = nwec.utility_reporting.arrearages.get_arrearages_df(
    df, NUM_MONTHS, COLS_PER_MONTH, ARREARAGE_SEARCH_STRING
)

# Arrearage Counts


In [3]:
date_row = nwec.utility_reporting.arrearages.infer_date_row(arrearage_counts, source_date_format)
arrearage_counts = arrearage_counts.tail(-date_row)  # remove rows before the date row
arrearage_counts = nwec.utility_reporting.arrearage_counts.format_arrearage_count_dates(
    arrearage_counts, source_date_format
)

In [4]:
# Need custom logic for finding the zip code column for PSE
customer_class_column = df.select(pl.nth(nwec.utility_reporting.arrearages.infer_customer_class_column(df)))
customer_class_row = nwec.utils.excel.find_cell_by_string(customer_class_column, "class")
customer_class_column = customer_class_column.rename({customer_class_column.columns[0]: "Customer Class"}).tail(
    -(customer_class_row[0] + 1)
)
zip_column = df.select(pl.nth(nwec.utils.excel.infer_zip_column(df)))
zip_column = zip_column.rename({zip_column.columns[0]: "Zip Code"}).tail(-(customer_class_row[0] + 1))
zip_column = zip_column.with_columns(pl.col("Zip Code").str.strip_chars())
arrearage_counts = pl.concat([zip_column, customer_class_column, arrearage_counts], how="horizontal")
arrearage_counts = nwec.utility_reporting.arrearage_counts.normalize_arrearage_count_cols(arrearage_counts, Utility.PSE)


/tmp/ipykernel_150278/3966739745.py:2: UserWarning: Multiple columns have at least 5 rows that match the customer class pattern; using the first.
  customer_class_column = df.select(pl.nth(nwec.utility_reporting.arrearages.infer_customer_class_column(df)))
/tmp/ipykernel_150278/3966739745.py:7: UserWarning: Multiple columns have at least 5 rows that match the ZIP code pattern; using the first.
  zip_column = df.select(pl.nth(nwec.utils.excel.infer_zip_column(df)))


# Save Results


In [5]:
nwec.utility_reporting.arrearage_counts.save_processed_arrearage_counts(arrearage_counts)